# UC-06 – Exploration

Notebook d'exploration du Jour 3. Il lit les sources, observe les schémas et les anomalies, et n'écrit rien.
Le nettoyage, Silver, Gold et Parquet sont dans `uc06_silver_gold.ipynb`.

Les fichiers n'ont pas d'en-tête. Les noms de colonnes sont ceux du schéma TPCx-BB, marqués **Déduit**, et ils ne sont posés que si la largeur observée correspond.
Les fichiers d'origine ne sont pas modifiés.

Exécuter les cellules dans l'ordre, dans le conteneur `lake` (Jupyter `http://localhost:8888/?token=tp`).


In [1]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("UC06-exploration")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Python", sys.version.split()[0])
print("Spark", spark.version)
print("mode", spark.sparkContext.master)
print("parallelisme", spark.sparkContext.defaultParallelism)
print("JAVA_HOME", os.environ.get("JAVA_HOME"))
print("coeurs", os.cpu_count())


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/19 09:53:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Python 3.12.3
Spark 4.3.0
mode local[*]
parallelisme 12
JAVA_HOME /usr/lib/jvm/java-17-openjdk-amd64
coeurs 12


## Sources retenues

Deux sources du lot initial, suffisantes pour la question métier du Jour 1 sur le chiffre d'affaires des ventes magasin :

- `store_sales.dat` : lignes de ticket magasin ;
- `item.dat` : référentiel produit, pour nommer les articles vendus.

Le lot refresh et les autres domaines ne sont pas relus ici.


In [2]:
VENTES = "/data/retail_project/data/store_sales/store_sales.dat"
PRODUITS = "/data/retail_project/data/item/item.dat"

VENTES_NOMS = [
    "ss_sold_date_sk", "ss_sold_time_sk", "ss_item_sk", "ss_customer_sk",
    "ss_cdemo_sk", "ss_hdemo_sk", "ss_addr_sk", "ss_store_sk", "ss_promo_sk",
    "ss_ticket_number", "ss_quantity", "ss_wholesale_cost", "ss_list_price",
    "ss_sales_price", "ss_ext_discount_amt", "ss_ext_sales_price",
    "ss_ext_wholesale_cost", "ss_ext_list_price", "ss_ext_tax", "ss_coupon_amt",
    "ss_net_paid", "ss_net_paid_inc_tax", "ss_net_profit",
]
PRODUITS_NOMS = [
    "i_item_sk", "i_item_id", "i_rec_start_date", "i_rec_end_date", "i_item_desc",
    "i_current_price", "i_wholesale_cost", "i_brand_id", "i_brand", "i_class_id",
    "i_class", "i_category_id", "i_category", "i_manufact_id", "i_manufact",
    "i_size", "i_formulation", "i_color", "i_units", "i_container",
    "i_manager_id", "i_product_name",
]

def lire_brut(chemin):
    return (
        spark.read
        .option("sep", "|")
        .option("header", False)
        .option("quote", "\u0000")
        .option("escape", "\u0000")
        .csv(chemin)
    )

def lignes_fichier(chemin):
    total = 0
    with open(chemin, "rb") as fichier:
        for _ in fichier:
            total += 1
    return total

sources = {"store_sales": (VENTES, VENTES_NOMS), "item": (PRODUITS, PRODUITS_NOMS)}
frames = {}
for nom, (chemin, colonnes) in sources.items():
    brut = lire_brut("file://" + chemin)
    largeur = len(brut.columns)
    print(f"{nom} : largeur observée {largeur}, largeur du schéma {len(colonnes)}")
    if largeur != len(colonnes):
        raise SystemExit(f"{nom} : largeur différente, noms Déduit non appliqués")
    frames[nom] = brut.toDF(*colonnes).persist()
    print(nom, "lignes fichier", lignes_fichier(chemin), "lignes spark", frames[nom].count())
    print(nom, "octets", os.path.getsize(chemin))


store_sales : largeur observée 23, largeur du schéma 23


store_sales lignes fichier 667579 lignes spark 667579
store_sales octets 92764700


item : largeur observée 22, largeur du schéma 22


item lignes fichier 17820 lignes spark 17820
item octets 6799286


## Schémas et échantillons

Lecture en texte : aucun type n'est inféré. Le schéma typé est choisi ensuite, à partir de ces valeurs.


In [3]:
for nom, df in frames.items():
    print("====", nom)
    df.printSchema()
    df.show(5, truncate=False)
    print("colonnes", len(df.columns), "lignes", df.count())


==== store_sales
root
 |-- ss_sold_date_sk: string (nullable = true)
 |-- ss_sold_time_sk: string (nullable = true)
 |-- ss_item_sk: string (nullable = true)
 |-- ss_customer_sk: string (nullable = true)
 |-- ss_cdemo_sk: string (nullable = true)
 |-- ss_hdemo_sk: string (nullable = true)
 |-- ss_addr_sk: string (nullable = true)
 |-- ss_store_sk: string (nullable = true)
 |-- ss_promo_sk: string (nullable = true)
 |-- ss_ticket_number: string (nullable = true)
 |-- ss_quantity: string (nullable = true)
 |-- ss_wholesale_cost: string (nullable = true)
 |-- ss_list_price: string (nullable = true)
 |-- ss_sales_price: string (nullable = true)
 |-- ss_ext_discount_amt: string (nullable = true)
 |-- ss_ext_sales_price: string (nullable = true)
 |-- ss_ext_wholesale_cost: string (nullable = true)
 |-- ss_ext_list_price: string (nullable = true)
 |-- ss_ext_tax: string (nullable = true)
 |-- ss_coupon_amt: string (nullable = true)
 |-- ss_net_paid: string (nullable = true)
 |-- ss_net_paid_i

+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+
|ss_sold_date_sk|ss_sold_time_sk|ss_item_sk|ss_customer_sk|ss_cdemo_sk|ss_hdemo_sk|ss_addr_sk|ss_store_sk|ss_promo_sk|ss_ticket_number|ss_quantity|ss_wholesale_cost|ss_list_price|ss_sales_price|ss_ext_discount_amt|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_list_price|ss_ext_tax|ss_coupon_amt|ss_net_paid|ss_net_paid_inc_tax|ss_net_profit|
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+

colonnes 23 lignes 667579
==== item
root
 |-- i_item_sk: string (nullable = true)
 |-- i_item_id: string (nullable = true)
 |-- i_rec_start_date: string (nullable = true)
 |-- i_rec_end_date: string (nullable = true)
 |-- i_item_desc: string (nullable = true)
 |-- i_current_price: string (nullable = true)
 |-- i_wholesale_cost: string (nullable = true)
 |-- i_brand_id: string (nullable = true)
 |-- i_brand: string (nullable = true)
 |-- i_class_id: string (nullable = true)
 |-- i_class: string (nullable = true)
 |-- i_category_id: string (nullable = true)
 |-- i_category: string (nullable = true)
 |-- i_manufact_id: string (nullable = true)
 |-- i_manufact: string (nullable = true)
 |-- i_size: string (nullable = true)
 |-- i_formulation: string (nullable = true)
 |-- i_color: string (nullable = true)
 |-- i_units: string (nullable = true)
 |-- i_container: string (nullable = true)
 |-- i_manager_id: string (nullable = true)
 |-- i_product_name: string (nullable = true)



+---------+----------------+----------------+--------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+----------------+----------+---------------------------------+----------+---------------------+-------------+-----------------+-------------+--------------------------------------------------+------+--------------------+--------+-------+-----------+------------+------------------------------------------------+
|i_item_sk|i_item_id       |i_rec_start_date|i_rec_end_date|i_item_desc                                                                                                                                                                                        |i_current_price|i_wholesale_cost|i_brand_id|i_brand                          |i_class_id|i_class              |i_category_id|i_category       |i_manufact_id|i_m

colonnes 22 lignes 17820


In [4]:
def vides(df):
    return df.select([
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])

for nom, df in frames.items():
    print("==== valeurs vides", nom)
    ligne = vides(df).collect()[0]
    total = df.count()
    for colonne in df.columns:
        nb = ligne[colonne]
        if nb:
            print(f"  {colonne} : {nb} ({nb / total:.2%})")
    if not any(ligne):
        print("  aucune valeur vide")


==== valeurs vides store_sales


  ss_cdemo_sk : 238 (0.04%)
  ss_hdemo_sk : 295 (0.04%)
  ss_addr_sk : 448 (0.07%)
  ss_promo_sk : 158 (0.02%)
==== valeurs vides item


  i_rec_end_date : 17820 (100.00%)
  i_item_desc : 4 (0.02%)
  i_brand_id : 8 (0.04%)
  i_brand : 6 (0.03%)


In [5]:
for nom, df in frames.items():
    total = df.count()
    distincts = df.distinct().count()
    print(nom, "lignes", total, "lignes distinctes", distincts, "doublons exacts", total - distincts)

ventes = frames["store_sales"]
produits = frames["item"]
print("tickets distincts", ventes.select("ss_ticket_number").distinct().count())
print("i_item_sk distincts", produits.select("i_item_sk").distinct().count())
print("i_item_id distincts", produits.select("i_item_id").distinct().count())

print("quantité min/max")
ventes.select(
    F.min("ss_quantity").alias("qte_min"),
    F.max("ss_quantity").alias("qte_max"),
).show()
print("ss_net_paid et ss_net_profit, aperçu des bornes (texte)")
ventes.select(
    F.min("ss_net_paid").alias("ca_min"),
    F.max("ss_net_paid").alias("ca_max"),
    F.min("ss_net_profit").alias("profit_min"),
    F.max("ss_net_profit").alias("profit_max"),
).show(truncate=False)


store_sales lignes 667579 lignes distinctes 667579 doublons exacts 0


item lignes 17820 lignes distinctes 17820 doublons exacts 0


tickets distincts 89100


i_item_sk distincts 17820


i_item_id distincts 17820
quantité min/max


+-------+-------+
|qte_min|qte_max|
+-------+-------+
|      1|     99|
+-------+-------+

ss_net_paid et ss_net_profit, aperçu des bornes (texte)


+------+-------+----------+----------+
|ca_min|ca_max |profit_min|profit_max|
+------+-------+----------+----------+
|0.00  |9999.82|-0.01     |999.96    |
+------+-------+----------+----------+



## Constats

Les deux fichiers sont structurés, séparés par `|`, sans en-tête. Tant que la lecture reste en texte, les types ne sont pas ceux du métier : les montants et les identifiants sont des chaînes. Le notebook de transformation pose un schéma explicite.

Granularité observée : une ligne de `store_sales` est une ligne de ticket (un article dans un ticket), pas un ticket. Une ligne de `item` est un article. `i_rec_end_date` est vide : aucune fin de version n'est observée.

Les champs vides de `store_sales` portent sur des clés secondaires (foyer, adresse, promotion), pas sur la date, l'article, le magasin, le ticket, la quantité ou le montant net. Un profit négatif, s'il apparaît, est conservé : il peut être un résultat de ligne, pas une erreur de fichier.
